# PV-Erzeugungsdaten: Datenqualitätsprüfung und Marktwert

**Praxisprojekt | Python · SQL (SQLite) · Plotly · PVGIS · SMARD**

Das Notebook bildet in vereinfachter Form ab, wie Erzeugungsdaten von PV-Anlagen in einem Energiedatenmanagement verarbeitet werden: Daten kommen an, werden auf Vollständigkeit und Plausibilität geprüft, Lücken werden vorläufig ersetzt, und erst danach werden die Daten ausgewertet.

**Aufbau**

| Schritt | Inhalt | Ergebnis |
|---|---|---|
| 1 | Einrichtung | Bibliotheken, Projektpfade |
| 2 | Daten laden | PV-Zeitreihen, Anlagenstammdaten, Kundenzuordnung, Day-Ahead-Preise, Eingangskontrolle |
| 3 | Lieferung simulieren | Kopie der Daten mit vier typischen Lieferfehlern |
| 4 | Daten prüfen | SQL-Prüfungen, Abgleich mit Soll-Zeitindex, Fehlerbericht, Tagesbericht |
| 5 | Ersatzwerte bilden | vorläufige Ersatzwerte, Validierung, Grafiken je Fehlerfall |
| 6 | Auswerten | Energie und Marktwert auf Basis der **geprüften** Daten (SQL) |
| 7 | Berichten | Kundendashboard, Export |

**Datenquellen:** PV-Erzeugung aus PVGIS (Joint Research Centre der EU, stündliche Simulation). Day-Ahead-Preise der Gebotszone DE-LU (EPEX SPOT), heruntergeladen über SMARD (Bundesnetzagentur). Kunden und Fehlerfälle sind fiktiv.

**Hinweis:** Der modellierte Marktwert ist weder Gewinn noch Abrechnung oder EEG-Vergütung. Ersatzwerte sind vorläufige Schätzungen und keine abrechnungsrelevanten Messwerte.

**Ausführung:** Notebook im Projektordner `Node Energie` in VS Code öffnen, den Kernel `.venv` auswählen und alle Zellen der Reihe nach ausführen.

## 1. Einrichtung

Die Pfade sind relativ zum Projektordner definiert. Erwartet werden `Daten/Lastgänge` (PVGIS), ein Preisordner (`Daten/Preise`; der bisherige Name `EEX_Presise` wird ebenfalls erkannt) und `Daten/Stammdaten`. Ergebnisse werden in getrennten Ausgabeordnern gespeichert.

In [ ]:
from pathlib import Path
import re
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

PROJECT_FOLDER = Path.cwd()
if not (PROJECT_FOLDER / "Daten").is_dir() and (PROJECT_FOLDER.parent / "Daten").is_dir():
    PROJECT_FOLDER = PROJECT_FOLDER.parent
DATA_FOLDER = PROJECT_FOLDER / "Daten"
PV_FOLDER = DATA_FOLDER / "Lastgänge"
PRICE_FOLDER = next((DATA_FOLDER / name for name in ("Preise", "EEX_Preise", "EEX_Presise")
                     if (DATA_FOLDER / name).is_dir()), DATA_FOLDER / "Preise")
KUNDEN_FILE = DATA_FOLDER / "Stammdaten" / "kunden_anlagen_zuordnung.csv"
if not KUNDEN_FILE.exists():
    KUNDEN_FILE = DATA_FOLDER / "kunden_anlagen_zuordnung.csv"

OUTPUT_FOLDER = DATA_FOLDER / "Export"
QUALITY_FOLDER = DATA_FOLDER / "Datenqualität"
for folder in (OUTPUT_FOLDER, QUALITY_FOLDER / "Testdaten",
               QUALITY_FOLDER / "Berichte", QUALITY_FOLDER / "Ersatzwerte"):
    folder.mkdir(parents=True, exist_ok=True)

TZ_LOKAL = "Europe/Berlin"
PRICE_SOURCE_COLUMN = "Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen"

pv_files = sorted(PV_FOLDER.glob("*.csv"))
price_files = sorted(PRICE_FOLDER.glob("*.csv"))
if not pv_files:
    raise FileNotFoundError(f"Keine PVGIS-CSV-Dateien in {PV_FOLDER}")
if len(price_files) != 1:
    raise ValueError(f"Erwartet wird genau eine Preis-CSV in {PRICE_FOLDER}; gefunden: {[p.name for p in price_files]}")
if not KUNDEN_FILE.exists():
    raise FileNotFoundError(f"Kundenzuordnung fehlt: {KUNDEN_FILE}")
PRICE_FILE = price_files[0]


def als_text(zeit):
    """Zeitstempel einheitlich als Text für SQLite formatieren."""
    return zeit.dt.strftime("%Y-%m-%d %H:%M:%S")


print("Projektordner:", PROJECT_FOLDER)
print("PV-Dateien:   ", [p.name for p in pv_files])
print("Preisdatei:   ", PRICE_FILE.name)
print("Kundendatei:  ", KUNDEN_FILE.name)

## 2. Daten laden
### 2.1 PVGIS-Zeitreihen

PVGIS-Dateien beginnen mit Metadaten; danach folgen die stündlichen Werte und am Ende erläuternde Textzeilen. Die Funktion sucht deshalb die eigentliche Kopfzeile und verwirft nicht-numerische Zeilen. PVGIS liefert UTC-Zeitstempel mit der Minute `:10`; sie werden der jeweiligen vollen Stunde zugeordnet (projektspezifische Annahme).

`P` ist die mittlere Leistung in W. Für ein einstündiges Intervall gilt: Energie in kWh = `P / 1000`.

In [ ]:
def find_header(file_path):
    """Zeilennummer der PVGIS-Kopfzeile zurückgeben."""
    with open(file_path, encoding="utf-8-sig") as file:
        for i, line in enumerate(file):
            if line.startswith("time,"):
                return i
    raise ValueError(f"PVGIS-Kopfzeile nicht gefunden: {file_path}")


def clean_pvgis(file_path):
    """Messwerte lesen, Erläuterungszeilen verwerfen, Zeitstempel nach UTC normieren."""
    df = pd.read_csv(file_path, skiprows=find_header(file_path))
    if not {"time", "P", "Gb(i)"}.issubset(df.columns):
        raise ValueError(f"PVGIS-Spalten fehlen in {file_path}")
    valid = (pd.to_numeric(df["P"], errors="coerce").notna()
             & pd.to_numeric(df["Gb(i)"], errors="coerce").notna())
    df = df.loc[valid].copy()
    df["time"] = pd.to_datetime(df["time"], format="%Y%m%d:%H%M")
    df["P"] = pd.to_numeric(df["P"])
    df["time_utc"] = df["time"].dt.floor("h").dt.tz_localize("UTC")
    return df[["time_utc", "P"]].sort_values("time_utc").reset_index(drop=True)


pv_original = pd.concat(
    [clean_pvgis(file).assign(Anlage=file.stem) for file in pv_files], ignore_index=True
).rename(columns={"P": "Power_W"})
pv_original["Energy_kWh"] = pv_original["Power_W"] / 1000   # 1-Stunden-Intervall
pv_original = (pv_original[["Anlage", "time_utc", "Power_W", "Energy_kWh"]]
               .sort_values(["Anlage", "time_utc"]).reset_index(drop=True))
JAHR = int(pv_original["time_utc"].dt.year.mode()[0])

print("Betrachtungsjahr:", JAHR)
display(pv_original.groupby("Anlage").size().rename("Zeilen").to_frame())

### 2.2 Anlagenstammdaten und Kundenzuordnung

Nennleistung, Koordinaten und Ausrichtung stehen im Kopf der PVGIS-Dateien. Die Nennleistung wird später für die Plausibilitätsprüfung gebraucht. Eine separate CSV-Datei ordnet die fiktiven Kunden den Standorten zu; eine Kontrolle stellt sicher, dass keine Anlage ohne Kunde bleibt.

In [ ]:
METADATA_NAMES = {
    "Latitude (decimal degrees)": "Latitude",
    "Longitude (decimal degrees)": "Longitude",
    "Elevation (m)": "Elevation_m",
    "Radiation database": "Radiation_source",
    "Slope": "Slope_deg",
    "Azimuth": "Azimuth_deg",
    "Nominal power of the PV system (c-Si) (kWp)": "Capacity_kWp",
    "System losses (%)": "System_losses_pct",
}


def get_metadata(file_path):
    metadata = {}
    with open(file_path, encoding="utf-8-sig") as file:
        for line in file:
            if line.strip().startswith("time,"):
                break
            if ":" in line:
                key, value = line.strip().rsplit(":", 1)
                key, value = key.strip(), value.replace("deg.", "").strip()
                if key in METADATA_NAMES:
                    column = METADATA_NAMES[key]
                    metadata[column] = value if column == "Radiation_source" else float(value)
    missing = set(METADATA_NAMES.values()) - set(metadata)
    if missing:
        raise ValueError(f"Metadaten fehlen in {file_path}: {sorted(missing)}")
    metadata["Anlage"] = Path(file_path).stem
    return metadata


def standortname(text):
    text = re.sub(r"(?i)(?:_pv)?_\d{4}$", "", str(text).strip())
    return text.replace("Muenchen", "München")


anlagen_stammdaten = pd.DataFrame([get_metadata(file) for file in pv_files])[
    ["Anlage", "Latitude", "Longitude", "Elevation_m", "Radiation_source",
     "Slope_deg", "Azimuth_deg", "Capacity_kWp", "System_losses_pct"]]
anlagen_stammdaten["Standort"] = anlagen_stammdaten["Anlage"].map(standortname)

kunden_anlagen = pd.read_csv(KUNDEN_FILE, sep=None, engine="python", encoding="utf-8-sig")
kunden_anlagen.columns = [c.strip().replace("Kunden-ID", "Kunde_ID").replace("Kunden_ID", "Kunde_ID")
                          for c in kunden_anlagen.columns]
if not {"Kunde_ID", "Anlage", "Kundenname"}.issubset(kunden_anlagen.columns):
    raise ValueError(f"Spalten Kunde_ID, Anlage, Kundenname erwartet. Vorhanden: {list(kunden_anlagen.columns)}")
kunden_anlagen["Standort"] = kunden_anlagen["Anlage"].map(standortname)

anlagen_mit_kunden = anlagen_stammdaten.merge(
    kunden_anlagen[["Kunde_ID", "Kundenname", "Standort"]], on="Standort", how="left", validate="one_to_one")
if anlagen_mit_kunden["Kundenname"].isna().any():
    raise ValueError("Kundenzuordnung fehlt für: " + ", ".join(
        anlagen_mit_kunden.loc[anlagen_mit_kunden["Kundenname"].isna(), "Standort"]))
display(anlagen_mit_kunden[["Kunde_ID", "Kundenname", "Standort", "Capacity_kWp",
                            "Latitude", "Longitude", "Slope_deg", "Azimuth_deg"]])

### 2.3 Day-Ahead-Preise

Die SMARD-Datei enthält die stündlichen Day-Ahead-Preise der Gebotszone DE-LU in deutscher Ortszeit. Für die Verknüpfung mit PVGIS werden sie nach UTC umgerechnet. Die doppelte Stunde beim Wechsel auf Winterzeit wird mit `ambiguous="infer"` aus der Reihenfolge der Zeilen aufgelöst.

In [ ]:
def clean_prices(file_path):
    prices = pd.read_csv(file_path, sep=";", decimal=",")
    required = {"Datum von", PRICE_SOURCE_COLUMN}
    if not required.issubset(prices.columns):
        raise ValueError(f"Spalten fehlen in der Preisdatei: {sorted(required - set(prices.columns))}")
    prices["Datum von"] = pd.to_datetime(prices["Datum von"], format="%d.%m.%Y %H:%M")
    prices["Price_EUR_MWh"] = pd.to_numeric(prices[PRICE_SOURCE_COLUMN], errors="coerce")
    # 'infer' setzt chronologische Zeilen voraus, die beide 02:00-Stunden enthalten.
    prices = prices.sort_values("Datum von", kind="stable").reset_index(drop=True)
    prices["time_utc"] = (prices["Datum von"]
        .dt.tz_localize(TZ_LOKAL, ambiguous="infer", nonexistent="raise")
        .dt.tz_convert("UTC"))
    prices = prices[["time_utc", "Price_EUR_MWh"]].copy()
    if prices["time_utc"].duplicated().any():
        raise ValueError("Doppelte UTC-Zeitstempel in den Preisen")
    return prices


prices = clean_prices(PRICE_FILE)
print("Preiswerte:", len(prices))
display(prices.head())

### 2.4 Eingangskontrolle der Quelldaten

Bevor die Daten weiterverwendet werden, wird geprüft, ob die Quelldaten selbst vollständig sind: jede Anlage gegen alle Stunden des Jahres in UTC (Soll-Zeitindex), dazu doppelte Zeitstempel, negative Werte und die Abdeckung durch Preise. Eine einzelne Stunde ohne Preis kann am Jahresende entstehen: Die letzte UTC-Stunde des Jahres liegt in deutscher Zeit bereits im Folgejahr.

In [ ]:
SOLL_INDEX = pd.date_range(f"{JAHR}-01-01 00:00", f"{JAHR}-12-31 23:00", freq="h", tz="UTC")

eingangskontrolle = pd.DataFrame([{
    "Anlage": name,
    "Zeilen": len(df),
    "Soll_Stunden": len(SOLL_INDEX),
    "Fehlende_Stunden": len(SOLL_INDEX.difference(df["time_utc"])),
    "Doppelte_Zeitstempel": int(df["time_utc"].duplicated().sum()),
    "Negative_Werte": int((df["Power_W"] < 0).sum()),
    "Stunden_ohne_Preis": len(SOLL_INDEX.difference(prices["time_utc"])),
} for name, df in pv_original.groupby("Anlage")])
eingangskontrolle.to_csv(OUTPUT_FOLDER / "datenqualitaet.csv", index=False)
display(eingangskontrolle)

## 3. Lieferung simulieren

Die PVGIS-Daten sind fehlerfrei. Um die Prüfung realistisch zu testen, wird eine **Kopie** erzeugt, die wie eine fehlerhafte Lieferung eines Messstellenbetreibers aussieht. Die Originalwerte bleiben unverändert und dienen später zur Validierung der Ersatzwerte.

| Standort | Datum (UTC) | eingebauter Fehler |
|---|---|---|
| Hamburg | 15.06., 10–12 Uhr | drei Zeilen fehlen komplett |
| München | 15.06., 10–11 Uhr | zwei unrealistisch hohe Werte |
| Dresden | 03.07., 12 Uhr | Zeitstempel doppelt geliefert, zweiter Wert weicht ab |
| Erfurt | 20.08., 0–2 Uhr | Erzeugung in der Nacht |

In [ ]:
lieferung = pv_original.copy(deep=True)
kapazitaet = anlagen_stammdaten.set_index("Anlage")["Capacity_kWp"]


def teststunden(ort, datum, stunden):
    """Zeilen eines Standorts zu bestimmten UTC-Stunden eines Tages auswählen."""
    maske = (lieferung["Anlage"].str.contains(ort, case=False, regex=True)
             & lieferung["time_utc"].dt.date.eq(pd.Timestamp(datum).date())
             & lieferung["time_utc"].dt.hour.isin(stunden))
    assert maske.sum() == len(stunden), f"Die erwarteten Teststunden fehlen: {ort} {datum}"
    return maske


# 1) Hamburg: drei Zeilen fehlen komplett
lieferung = lieferung.loc[~teststunden("Hamburg", f"{JAHR}-06-15", [10, 11, 12])]

# 2) München: zwei unrealistisch hohe Werte
muenchen = teststunden("München|Muenchen", f"{JAHR}-06-15", [10, 11])
lieferung.loc[muenchen, "Power_W"] = [1_450_000, 1_600_000]
lieferung.loc[muenchen, "Energy_kWh"] = [1450, 1600]

# 3) Dresden: Zeitstempel doppelt geliefert, zweiter Wert weicht ab
duplikat = lieferung.loc[teststunden("Dresden", f"{JAHR}-07-03", [12])].copy()
duplikat[["Power_W", "Energy_kWh"]] *= 0.8
lieferung = pd.concat([lieferung, duplikat], ignore_index=True)

# 4) Erfurt: Erzeugung in der Nacht (5 % der Nennleistung)
erfurt = teststunden("Erfurt", f"{JAHR}-08-20", [0, 1, 2])
erfurt_kwp = kapazitaet[lieferung.loc[erfurt, "Anlage"].iloc[0]]
lieferung.loc[erfurt, "Energy_kWh"] = 0.05 * erfurt_kwp
lieferung.loc[erfurt, "Power_W"] = 0.05 * erfurt_kwp * 1000

lieferung = lieferung.sort_values(["Anlage", "time_utc"], kind="stable").reset_index(drop=True)
lieferung.to_csv(QUALITY_FOLDER / "Testdaten" / "pv_erzeugung_fehlerhaft.csv", index=False)

testfaelle = pd.DataFrame([
    {"Standort": "Hamburg", "Datum": f"{JAHR}-06-15", "Fehlerart": "Fehlender Wert"},
    {"Standort": "München", "Datum": f"{JAHR}-06-15", "Fehlerart": "Leistung zu hoch"},
    {"Standort": "Dresden", "Datum": f"{JAHR}-07-03", "Fehlerart": "Doppelter Zeitstempel"},
    {"Standort": "Erfurt",  "Datum": f"{JAHR}-08-20", "Fehlerart": "Erzeugung in der Nacht"},
])
print("Zeilen im Original:", len(pv_original), "| Zeilen in der Lieferung:", len(lieferung))
display(testfaelle)

## 4. Daten prüfen
### 4.1 Daten in SQLite laden

Stammdaten, Kundenzuordnung und die Lieferung werden in SQLite gespeichert. Zeitstempel liegen dort einheitlich als Text vor: in UTC für Abgleich und Prüfung, zusätzlich in deutscher Ortszeit für Tages- und Monatsberichte.

In [ ]:
conn = sqlite3.connect(PROJECT_FOLDER / "PV_Portfolio.db")

anlagen_mit_kunden.to_sql("Anlagen_Kunden", conn, if_exists="replace", index=False)

lieferung_sql = lieferung.copy()
lieferung_sql["time_local"] = als_text(lieferung_sql["time_utc"].dt.tz_convert(TZ_LOKAL))
lieferung_sql["time_utc"] = als_text(lieferung_sql["time_utc"])
lieferung_sql.to_sql("Lieferung", conn, if_exists="replace", index=False)

print("SQLite-Tabellen:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")])

### 4.2 Prüfung mit SQL: doppelte Zeitstempel und Lücken

- **Doppelte Zeitstempel:** `GROUP BY … HAVING COUNT(*) > 1` findet Stunden, die mehrfach geliefert wurden.
- **Lücken:** Die Fensterfunktion `LAG()` vergleicht jeden Zeitstempel mit dem vorherigen derselben Anlage. Liegt mehr als eine Stunde dazwischen, fehlen Zeilen.

In [ ]:
doppelte_sql = pd.read_sql_query("""
SELECT Anlage, time_utc, COUNT(*) AS Anzahl,
       GROUP_CONCAT(ROUND(Energy_kWh, 2), ' / ') AS Gelieferte_Werte_kWh
FROM Lieferung
GROUP BY Anlage, time_utc
HAVING COUNT(*) > 1
""", conn)

luecken_sql = pd.read_sql_query("""
WITH folge AS (
    SELECT Anlage, time_utc,
           LAG(time_utc) OVER (PARTITION BY Anlage ORDER BY time_utc) AS vorher
    FROM (SELECT DISTINCT Anlage, time_utc FROM Lieferung)
)
SELECT Anlage,
       vorher   AS Letzter_Wert_vor_Luecke,
       time_utc AS Erster_Wert_nach_Luecke,
       CAST(ROUND((julianday(time_utc) - julianday(vorher)) * 24) AS INTEGER) - 1 AS Fehlende_Stunden
FROM folge
WHERE (julianday(time_utc) - julianday(vorher)) * 24 > 1.5
""", conn)

print("Doppelte Zeitstempel:")
display(doppelte_sql)
print("Datenlücken:")
display(luecken_sql)

### 4.3 Abgleich mit dem Soll-Zeitindex und Plausibilitätsregeln

Jede Anlage wird gegen alle Stunden des Jahres geprüft. Fehlende Zeilen werden dadurch zu leeren Werten und sind erkennbar, auch am Anfang oder Ende des Zeitraums. Doppelte Zeitstempel mit widersprüchlichen Werten gelten als ungültig.

Regeln für Einzelwerte:
- **Leistung zu hoch:** über 120 % der Nennleistung (vereinfachte Projektschwelle, keine technische oder regulatorische Grenze)
- **Negativer Wert**
- **Erzeugung in der Nacht:** Wert > 0 in einer Stunde, in der die Anlage laut Simulation im ganzen Monat nie erzeugt. In der Praxis würde man dafür den Sonnenstand berechnen (z. B. mit `pvlib`).

Ein gültiger Nullwert in der Nacht gilt **nicht** als Datenlücke. Am Ende prüft ein Selbsttest, ob alle eingebauten Fehler erkannt wurden.

In [ ]:
soll = pd.MultiIndex.from_product([anlagen_stammdaten["Anlage"], SOLL_INDEX],
                                  names=["Anlage", "time_utc"]).to_frame(index=False)

doppelt_maske = lieferung.duplicated(["Anlage", "time_utc"], keep=False)
doppelte_zeiten = (lieferung.loc[doppelt_maske, ["Anlage", "time_utc"]]
                   .drop_duplicates().assign(Doppelt=True))
pruefdaten = (soll
    .merge(lieferung.loc[~doppelt_maske], on=["Anlage", "time_utc"], how="left", validate="one_to_one")
    .merge(doppelte_zeiten, on=["Anlage", "time_utc"], how="left", validate="one_to_one"))
pruefdaten["Doppelt"] = pruefdaten["Doppelt"].eq(True)

# Nachtstunden: Monat und UTC-Stunde, in denen die Anlage laut Simulation nie erzeugt
nachtstunden = (pv_original
    .assign(Monat=pv_original["time_utc"].dt.month, Stunde=pv_original["time_utc"].dt.hour)
    .groupby(["Anlage", "Monat", "Stunde"], as_index=False)["Energy_kWh"].max()
    .assign(Nachtstunde=lambda d: d["Energy_kWh"].eq(0))
    .drop(columns="Energy_kWh"))
pruefdaten = (pruefdaten
    .assign(Monat=pruefdaten["time_utc"].dt.month, Stunde=pruefdaten["time_utc"].dt.hour)
    .merge(nachtstunden, on=["Anlage", "Monat", "Stunde"], how="left", validate="many_to_one")
    .drop(columns=["Monat", "Stunde"]))
pruefdaten["Nachtstunde"] = pruefdaten["Nachtstunde"].eq(True)

pruefdaten = pruefdaten.merge(
    anlagen_mit_kunden[["Anlage", "Standort", "Kundenname", "Capacity_kWp"]],
    on="Anlage", how="left", validate="many_to_one")


def pruefe_datenqualitaet(daten, grenzfaktor=1.20):
    """Markiert doppelte, fehlende, negative, zu hohe und nächtliche Erzeugungswerte."""
    ergebnis = daten.copy()
    ergebnis["Fehlend"] = ergebnis["Energy_kWh"].isna() & ~ergebnis["Doppelt"]
    ergebnis["Zu_hoch"] = ergebnis["Power_W"] > ergebnis["Capacity_kWp"] * 1000 * grenzfaktor
    ergebnis["Negativ"] = ergebnis["Energy_kWh"] < 0
    ergebnis["Nachtwert"] = ergebnis["Nachtstunde"] & (ergebnis["Energy_kWh"] > 0)
    ergebnis["Pruefung_erforderlich"] = ergebnis[
        ["Doppelt", "Fehlend", "Zu_hoch", "Negativ", "Nachtwert"]].any(axis=1)
    ergebnis["Fehlerart"] = np.select(
        [ergebnis["Doppelt"], ergebnis["Fehlend"], ergebnis["Zu_hoch"],
         ergebnis["Negativ"], ergebnis["Nachtwert"]],
        ["Doppelter Zeitstempel", "Fehlender Wert", "Leistung zu hoch",
         "Negativer Wert", "Erzeugung in der Nacht"], default="")
    return ergebnis


pruefdaten = pruefe_datenqualitaet(pruefdaten)

fehlerbericht = pruefdaten.loc[pruefdaten["Pruefung_erforderlich"], [
    "Kundenname", "Standort", "Anlage", "time_utc", "Fehlerart",
    "Power_W", "Energy_kWh", "Capacity_kWp"]].sort_values(["Standort", "time_utc"])
fehlerbericht.to_csv(QUALITY_FOLDER / "Berichte" / "datenqualitaetsbericht.csv", index=False, encoding="utf-8-sig")

print(f"Erkannte Fehler: {len(fehlerbericht)}")
display(fehlerbericht["Fehlerart"].value_counts().rename("Anzahl").to_frame())
display(fehlerbericht)

# Selbsttest: Wurde jeder eingebaute Fehler am richtigen Tag erkannt?
testfaelle["Gefunden"] = [
    ((fehlerbericht["Standort"] == fall.Standort)
     & (fehlerbericht["Fehlerart"] == fall.Fehlerart)
     & (fehlerbericht["time_utc"].dt.date == pd.Timestamp(fall.Datum).date())).any()
    for fall in testfaelle.itertuples()
]
display(testfaelle)
assert testfaelle["Gefunden"].all(), "Nicht alle Testfälle wurden erkannt."

### 4.4 Tagesbericht

Pro Kunde, Standort und deutschem Kalendertag: erwartete Stunden, fehlende und auffällige Werte. Der Tag der Umstellung auf Sommerzeit hat 23, der Tag der Umstellung auf Winterzeit 25 Soll-Stunden. Der 01.01. hat ebenfalls nur 23 Stunden, weil PVGIS ein UTC-Jahr liefert.

Der Status leitet die nächste Aktion ab: Fehlen Werte, werden sie beim Messstellenbetreiber (MSB) angefordert; sind alle Werte da, aber einzelne auffällig, ist eine Prüfung nötig.

In [ ]:
pruefdaten["Datum"] = pruefdaten["time_utc"].dt.tz_convert(TZ_LOKAL).dt.date
tagesbericht = pruefdaten.groupby(["Kundenname", "Standort", "Datum"], as_index=False).agg(
    Soll_Stunden=("time_utc", "size"),
    Fehlende_Werte=("Fehlend", "sum"),
    Doppelte_Zeitstempel=("Doppelt", "sum"),
    Unplausible_Werte=("Zu_hoch", "sum"),
    Negative_Werte=("Negativ", "sum"),
    Nachtwerte=("Nachtwert", "sum"),
)
tagesbericht = tagesbericht.loc[pd.to_datetime(tagesbericht["Datum"]).dt.year == JAHR].reset_index(drop=True)
tagesbericht["Status"] = np.select(
    [tagesbericht[["Fehlende_Werte", "Doppelte_Zeitstempel"]].sum(axis=1) > 0,
     tagesbericht[["Unplausible_Werte", "Negative_Werte", "Nachtwerte"]].sum(axis=1) > 0],
    ["Daten beim MSB anfordern", "Prüfung erforderlich"], default="OK")
tagesbericht.to_csv(QUALITY_FOLDER / "Berichte" / "tagesbericht.csv", index=False, encoding="utf-8-sig")

zeitumstellung = pd.to_datetime(tagesbericht["Datum"]).dt.month.isin([3, 10]) & (tagesbericht["Soll_Stunden"] != 24)
print("Tage mit Zeitumstellung:")
display(tagesbericht.loc[zeitumstellung, ["Datum", "Soll_Stunden"]].drop_duplicates())
print("Tage mit Handlungsbedarf:")
display(tagesbericht.loc[tagesbericht["Status"] != "OK"])

## 5. Ersatzwerte bilden
### 5.1 Vorläufige Ersatzwerte

Für jeden fehlenden oder auffälligen Wert werden **gültige Werte derselben UTC-Stunde** am Vortag und Folgetag herangezogen; der Ersatzwert ist deren Mittelwert. Fehlt einer der beiden Referenzwerte, bleibt der Wert offen und erhält den Status „Manuelle Prüfung erforderlich“. Die gelieferten Werte werden nicht überschrieben. Das ist eine einfache analytische Schätzung, **keine abrechnungsrelevante Ersatzwertbildung**.

In [ ]:
korrekturdaten = pruefdaten.sort_values(["Anlage", "time_utc"]).reset_index(drop=True).copy()
fehler_mask = korrekturdaten["Pruefung_erforderlich"]
referenz = korrekturdaten.loc[~fehler_mask, ["Anlage", "time_utc", "Energy_kWh"]]

vortag = referenz.rename(columns={"Energy_kWh": "Vortag_kWh"}).assign(
    time_utc=lambda d: d["time_utc"] + pd.Timedelta(hours=24))
folgetag = referenz.rename(columns={"Energy_kWh": "Folgetag_kWh"}).assign(
    time_utc=lambda d: d["time_utc"] - pd.Timedelta(hours=24))
korrekturdaten = (korrekturdaten
    .merge(vortag, on=["Anlage", "time_utc"], how="left", validate="one_to_one")
    .merge(folgetag, on=["Anlage", "time_utc"], how="left", validate="one_to_one"))

beide_vorhanden = fehler_mask & korrekturdaten["Vortag_kWh"].notna() & korrekturdaten["Folgetag_kWh"].notna()
korrekturdaten["Ersatzwert_kWh"] = np.where(
    beide_vorhanden, (korrekturdaten["Vortag_kWh"] + korrekturdaten["Folgetag_kWh"]) / 2, np.nan)
korrekturdaten["Energy_kWh_korrigiert"] = np.where(
    fehler_mask, korrekturdaten["Ersatzwert_kWh"], korrekturdaten["Energy_kWh"])
korrekturdaten["Datenstatus"] = np.select(
    [~fehler_mask, beide_vorhanden], ["gemessen", "vorläufig geschätzt"],
    default="Manuelle Prüfung erforderlich")

ersatzwertbericht = korrekturdaten.loc[fehler_mask, [
    "Kundenname", "Standort", "Anlage", "time_utc", "Fehlerart", "Energy_kWh",
    "Vortag_kWh", "Folgetag_kWh", "Ersatzwert_kWh", "Datenstatus"]].copy()
ersatzwertbericht["Methode"] = "Mittelwert Vortag/Folgetag (gleiche UTC-Stunde)"
ersatzwertbericht.to_csv(QUALITY_FOLDER / "Ersatzwerte" / "ersatzwertbericht.csv", index=False, encoding="utf-8-sig")

print(f"Ersatzwerte verfügbar: {beide_vorhanden.sum()} von {fehler_mask.sum()}")
display(ersatzwertbericht.drop(columns=["Anlage", "Methode"]))

### 5.2 Ersatzwerte gegen das Original validieren

Weil die Fehler künstlich eingebaut wurden, sind die ursprünglichen PVGIS-Werte bekannt. Damit lässt sich die Abweichung jedes Ersatzwerts messen; als Kennzahl dient der mittlere absolute Fehler (MAE) in kWh. Bei so wenigen Testwerten ist das ein Plausibilitätscheck, keine allgemeine Bewertung des Verfahrens.

Die Methode ist schwach, wenn sich das Wetter an Vortag, Fehlertag und Folgetag stark unterscheidet. Eine bessere Methode würde Vergleichsanlagen in der Nähe oder Einstrahlungsdaten nutzen.

In [ ]:
original_kwh = pv_original[["Anlage", "time_utc", "Energy_kWh"]].rename(columns={"Energy_kWh": "Original_kWh"})
vergleich = ersatzwertbericht.merge(original_kwh, on=["Anlage", "time_utc"], how="left", validate="one_to_one")
vergleich["Abweichung_kWh"] = vergleich["Ersatzwert_kWh"] - vergleich["Original_kWh"]
vergleich["Absoluter_Fehler_kWh"] = vergleich["Abweichung_kWh"].abs()

display(vergleich[["Standort", "time_utc", "Fehlerart", "Original_kWh", "Ersatzwert_kWh",
                   "Abweichung_kWh"]].round({"Original_kWh": 2, "Ersatzwert_kWh": 2, "Abweichung_kWh": 2}))
print(f"MAE der Ersatzwerte: {vergleich['Absoluter_Fehler_kWh'].mean():.2f} kWh")
vergleich.to_csv(QUALITY_FOLDER / "Berichte" / "validierung_ersatzwerte.csv", index=False, encoding="utf-8-sig")

### 5.3 Fehlerfälle grafisch prüfen

Je Fehlerfall zeigt eine Grafik den betroffenen Tag: die unveränderte PVGIS-Kurve, die fehlerhafte Lieferung und den Verlauf nach der Ersatzwertbildung.

In [ ]:
plot_df = korrekturdaten.merge(original_kwh, on=["Anlage", "time_utc"], how="left", validate="one_to_one")

for fall in testfaelle.itertuples():
    tag = pd.Timestamp(fall.Datum, tz="UTC")
    daten = plot_df.loc[(plot_df["Standort"] == fall.Standort)
                        & (plot_df["time_utc"] >= tag)
                        & (plot_df["time_utc"] < tag + pd.Timedelta(days=1))].sort_values("time_utc")
    fehler = daten["Pruefung_erforderlich"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=daten["time_utc"], y=daten["Original_kWh"],
                             mode="lines", name="Original (PVGIS)", line=dict(width=3)))
    fig.add_trace(go.Scatter(x=daten["time_utc"], y=daten["Energy_kWh"],
                             mode="lines+markers", name="Fehlerhafte Lieferung",
                             line=dict(width=2, dash="dash"), connectgaps=False))
    fig.add_trace(go.Scatter(x=daten["time_utc"], y=daten["Energy_kWh_korrigiert"],
                             mode="lines", name="Nach Ersatzwertbildung", line=dict(width=3, dash="dot")))
    fig.add_trace(go.Scatter(x=daten.loc[fehler, "time_utc"], y=daten.loc[fehler, "Ersatzwert_kWh"],
                             mode="markers", name="Ersatzwert", marker=dict(size=11, symbol="diamond")))
    fig.update_layout(title=f"{fall.Standort}, {tag:%d.%m.%Y}: {fall.Fehlerart}",
                      xaxis_title="Uhrzeit (UTC)", yaxis_title="Erzeugung (kWh)",
                      template="plotly_white", width=900, height=460, hovermode="x unified",
                      legend=dict(orientation="h", y=-0.28), margin=dict(t=60, b=100, l=65, r=20))
    fig.update_xaxes(tickformat="%H:%M", dtick=2 * 60 * 60 * 1000)
    fig.show()
    fig.write_html(OUTPUT_FOLDER / f"datenqualitaet_{fall.Standort}_{fall.Datum}.html", include_plotlyjs=True)

## 6. Auswerten auf Basis der geprüften Daten
### 6.1 Bereinigte Zeitreihe mit Preisen verknüpfen

Ausgewertet wird erst jetzt, und zwar mit den **geprüften** Werten: gemessene Werte, wo sie plausibel sind, sonst die vorläufigen Ersatzwerte. Jede Stunde behält ihren Datenstatus, damit sichtbar bleibt, wie viel einer Summe geschätzt ist.

Energie × Day-Ahead-Preis ergibt einen geschätzten Großhandelsmarktwert. Fehlende Preise bleiben leer und werden nicht als 0 € gewertet.

In [ ]:
erzeugung_bereinigt = (korrekturdaten[["Kundenname", "Standort", "Anlage", "time_utc", "Datenstatus"]]
    .assign(Energy_kWh=korrekturdaten["Energy_kWh_korrigiert"])
    .merge(prices, on="time_utc", how="left", validate="many_to_one"))
erzeugung_bereinigt["Revenue_EUR"] = erzeugung_bereinigt["Energy_kWh"] / 1000 * erzeugung_bereinigt["Price_EUR_MWh"]

sql_bereinigt = erzeugung_bereinigt.copy()
sql_bereinigt["time_local"] = als_text(sql_bereinigt["time_utc"].dt.tz_convert(TZ_LOKAL))
sql_bereinigt["time_utc"] = als_text(sql_bereinigt["time_utc"])
sql_bereinigt.to_sql("Erzeugung_bereinigt", conn, if_exists="replace", index=False)

display(erzeugung_bereinigt["Datenstatus"].value_counts().rename("Stunden").to_frame())

### 6.2 Jahres- und Monatswerte mit SQL

Gruppiert wird nach deutscher Ortszeit, so wie Kunden und Netzbetreiber abrechnen. Neben Energie und Marktwert berechnet SQL:
- **Marktwert je MWh:** Summe Erlös ÷ Summe Energie, nur Stunden mit Preis. Liegt er unter dem Durchschnittspreis, erzeugt die Anlage vor allem in günstigen Stunden – für PV typisch, weil viele Anlagen gleichzeitig mittags einspeisen.
- **Spezifischer Ertrag (kWh/kWp):** macht Anlagen unterschiedlicher Größe vergleichbar.
- **Anzahl geschätzter Stunden:** zeigt, wie stark ein Ergebnis auf Ersatzwerten beruht.

In [ ]:
jahresbericht = pd.read_sql_query("""
SELECT e.Kundenname, e.Standort,
       MAX(a.Capacity_kWp)                                          AS Leistung_kWp,
       ROUND(SUM(e.Energy_kWh) / 1000, 1)                           AS Erzeugung_MWh,
       ROUND(SUM(e.Energy_kWh) / MAX(a.Capacity_kWp), 0)            AS Spezifischer_Ertrag_kWh_kWp,
       ROUND(SUM(e.Revenue_EUR), 0)                                 AS Marktwert_EUR,
       ROUND(1000 * SUM(e.Revenue_EUR)
             / SUM(CASE WHEN e.Price_EUR_MWh IS NOT NULL THEN e.Energy_kWh END), 2)
                                                                    AS Marktwert_EUR_MWh,
       ROUND(AVG(e.Price_EUR_MWh), 2)                               AS Durchschnittspreis_EUR_MWh,
       SUM(e.Datenstatus = 'vorläufig geschätzt')                   AS Geschaetzte_Stunden,
       SUM(e.Datenstatus = 'Manuelle Prüfung erforderlich')         AS Offene_Stunden
FROM Erzeugung_bereinigt e
JOIN Anlagen_Kunden a USING (Anlage)
WHERE strftime('%Y', e.time_local) = :jahr
GROUP BY e.Kundenname, e.Standort
ORDER BY e.Standort
""", conn, params={"jahr": str(JAHR)})
display(jahresbericht)

monatsbericht = pd.read_sql_query("""
SELECT Kundenname, Standort,
       strftime('%Y-%m', time_local)                AS Monat,
       ROUND(SUM(Energy_kWh), 1)                    AS Energie_kWh,
       ROUND(SUM(Revenue_EUR), 2)                   AS Marktwert_EUR,
       SUM(Datenstatus = 'vorläufig geschätzt')     AS Geschaetzte_Stunden
FROM Erzeugung_bereinigt
WHERE strftime('%Y', time_local) = :jahr
GROUP BY Kundenname, Standort, Monat
ORDER BY Standort, Monat
""", conn, params={"jahr": str(JAHR)})
monatsbericht.to_csv(OUTPUT_FOLDER / "monthly_report.csv", index=False, encoding="utf-8-sig")
display(monatsbericht.loc[monatsbericht["Geschaetzte_Stunden"] > 0])

## 7. Berichten
### 7.1 Portfolioübersicht

Monatliche Erzeugung aller Anlagen auf Basis der geprüften Daten.

In [ ]:
fig_monat = px.line(monatsbericht, x="Monat", y="Energie_kWh", color="Standort", markers=True,
                    title=f"PV-Portfolio – Monatserzeugung {JAHR} (geprüfte Daten)",
                    labels={"Energie_kWh": "Erzeugung (kWh)", "Standort": "Anlage"})
fig_monat.update_layout(template="plotly_white")
fig_monat.show()

### 7.2 Kundenbezogenes Dashboard

Interaktive Auswertung je Kunde mit Filtern für Anlagen, Zeitraum, Auflösung und Kennzahl. Benötigt `ipywidgets`; falls die Widgets in VS Code nicht angezeigt werden, genügt die Übersicht oben.

In [ ]:
df = erzeugung_bereinigt.copy()
df["time_local"] = df["time_utc"].dt.tz_convert(TZ_LOKAL)
df["Date"] = df["time_local"].dt.date
kunden_daten = anlagen_mit_kunden[["Kundenname", "Standort", "Anlage"]]

try:
    import ipywidgets as widgets
    kunde = widgets.Dropdown(options=sorted(kunden_daten["Kundenname"].unique()), description="Kunde:")
    anlagen = widgets.SelectMultiple(options=[], rows=4, description="Anlagen:")
    startdatum = widgets.DatePicker(description="Von:", value=df["Date"].min())
    enddatum = widgets.DatePicker(description="Bis:", value=df["Date"].max())
    zeitraum = widgets.Dropdown(options=["Stündlich", "Täglich", "Monatlich", "Jährlich"],
                                value="Monatlich", description="Zeitraum:")
    kennzahl = widgets.Dropdown(options={"Erzeugung (kWh)": "Energy_kWh",
                                         "Marktwert (€)": "Revenue_EUR"}, description="Kennzahl:")

    def aktualisiere_anlagen(change=None):
        liste = kunden_daten.loc[kunden_daten["Kundenname"] == kunde.value, "Anlage"].tolist()
        anlagen.options = [(standortname(name), name) for name in liste]
        anlagen.value = tuple(liste)

    def zeichne_dashboard(selected_anlagen, start, end, periode, metric):
        if not selected_anlagen or start is None or end is None or start > end:
            print("Bitte Anlagen und einen gültigen Zeitraum auswählen.")
            return
        daten = df.loc[df["Anlage"].isin(selected_anlagen) & df["Date"].between(start, end)].copy()
        if daten.empty:
            print("Keine Daten für diese Auswahl.")
            return
        frequenz = {"Stündlich": "h", "Täglich": "D", "Monatlich": "M", "Jährlich": "Y"}[periode]
        daten["Periode"] = daten["time_local"].dt.tz_localize(None).dt.to_period(frequenz).dt.start_time
        agg = daten.groupby(["Standort", "Periode"], as_index=False)[metric].sum(min_count=1)
        labels = {"Energy_kWh": "Erzeugung (kWh)", "Revenue_EUR": "Modellierter Marktwert (€)"}
        if periode == "Jährlich":
            agg["Jahr"] = agg["Periode"].dt.year.astype(str)
            fig = px.bar(agg, x="Jahr", y=metric, color="Standort", barmode="group",
                         title=f"{kunde.value}: Jahresvergleich", labels={metric: labels[metric]})
        else:
            fig = px.line(agg, x="Periode", y=metric, color="Standort", markers=(periode == "Monatlich"),
                          title=f"{kunde.value}: {periode}", labels={metric: labels[metric]})
        fig.update_layout(template="plotly_white", height=470, hovermode="x unified")
        fig.show()

    kunde.observe(aktualisiere_anlagen, names="value")
    aktualisiere_anlagen()
    dashboard_output = widgets.interactive_output(zeichne_dashboard, {
        "selected_anlagen": anlagen, "start": startdatum, "end": enddatum,
        "periode": zeitraum, "metric": kennzahl})
    display(widgets.VBox([kunde, anlagen, widgets.HBox([startdatum, enddatum]), zeitraum, kennzahl]),
            dashboard_output)
except ImportError as exc:
    print("Interaktive Steuerelemente nicht verfügbar:", exc)

### 7.3 Export

Stammdaten, geprüfte Zeitreihe und Jahresbericht werden als CSV gespeichert. Zeitstempel werden im ISO-Format mit `Z` (UTC) geschrieben, damit sie in anderen Programmen eindeutig sind.

In [ ]:
export_bereinigt = erzeugung_bereinigt.copy()
export_bereinigt["time_utc"] = export_bereinigt["time_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

anlagen_mit_kunden.to_csv(OUTPUT_FOLDER / "anlagen_stammdaten.csv", index=False, encoding="utf-8-sig")
export_bereinigt.to_csv(OUTPUT_FOLDER / "anlagen_erzeugung.csv", index=False, encoding="utf-8-sig")
jahresbericht.to_csv(OUTPUT_FOLDER / "jahresbericht.csv", index=False, encoding="utf-8-sig")

conn.close()
print("Ergebnisse gespeichert in:", OUTPUT_FOLDER, "und", QUALITY_FOLDER)
print("Datenbankverbindung geschlossen.")

## Abschluss und Grenzen

**Ergebnis:** Das Notebook prüft eine simulierte Lieferung von PV-Erzeugungsdaten gegen einen vollständigen Soll-Zeitindex und mit SQL-Regeln. Alle vier eingebauten Fehlerarten werden erkannt, im Tagesbericht mit einer nächsten Aktion versehen und vorläufig ersetzt. Energie und Marktwert werden erst danach und auf Basis der geprüften Daten berechnet; geschätzte Stunden bleiben in jeder Summe sichtbar.

**Grenzen:**
- Die Daten sind stündlich und simuliert; reale Erzeugungsmesswerte liegen meist als Viertelstundenwerte vor.
- Reale Messwerte kommen über die Marktkommunikation (z. B. MSCONS) mit Marktlokations-IDs und Korrekturlieferungen.
- Schwellenwerte und Nacht-Referenz sind vereinfachte Projektannahmen.
- Die Ersatzwertmethode ist bei wechselndem Wetter ungenau (siehe 5.2) und nicht abrechnungsrelevant.

**Ergebnisdateien:** `Daten/Export` (Auswertungen, Grafiken) und `Daten/Datenqualität` (fehlerhafte Lieferung, Fehler- und Tagesbericht, Ersatzwerte, Validierung).